# VEA Algorithm: Experiment Pipeline

This notebook demonstrates the complete extraction pipeline for processing videos using the **VEA_algo** framework. It simulates the following flow:

1. **Load Video**
2. **Detect Scenes** (using PySceneDetect)
3. **Extract per scene** using a Vision Language Model (**Qwen3-VL-2B-Instruct** with `flash_attention_2` and `presence_penalty=1.5`) and an ASR model (**Whisper** with `repetition_penalty=2.0` / penalty factor for Text).
4. **Merge to Corpus** (Combine Visual Captions and Transcripts into a structured JSON dataset)

Let's import the necessary modules from the Extraction Module (`vea_extraction.py`).

In [ ]:
import os
import json
import logging
import traceback
from IPython.display import display, JSON

# Set up basic logging for the notebook
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('VEA_Experiment')

# Import the core extraction classes from our custom module
try:
    from modules.scene_detector import SceneExtractor
    from modules.captioner import VLMProcessor
    from modules.transcriber import ASRProcessor
    from utils.data_handler import save_corpus, cleanup_temp_dir
    logger.info("Successfully imported VEA extraction modules.")
except ImportError as e:
    logger.error(f"Failed to import modules. Ensure the modules directory exists. Error: {e}")

## 1. Load Video & Initialize Extractors

First, we define our input video path and initialize our VLM and ASR processors. The VLM automatically uses `flash_attention_2` to optimize memory usage.

In [ ]:
# Specify the path to your test video
VIDEO_PATH = "../VideoRAG/movies/example.mp4" # Replace with an actual video file path
OUTPUT_JSON = "output_corpus.json"

if not os.path.exists(VIDEO_PATH):
    logger.warning(f"Target video '{VIDEO_PATH}' not found. Please provide a valid video path for testing.")

# Initialize extractors with explicit exception handling
try:
    logger.info("Initializing Module: SceneExtractor")
    scene_extractor = SceneExtractor(output_dir="./vea_temp_scenes")
    
    logger.info("Initializing Module: VLM (Qwen3-VL-2B-Instruct)")
    vlm_processor = VLMProcessor(model_id="Qwen/Qwen2-VL-2B-Instruct")
    
    logger.info("Initializing Module: ASR (Faster-Whisper)")
    asr_processor = ASRProcessor(model_size="distil-large-v3")
    
except Exception as e:
    logger.error(f"Initialization error: {e}")
    traceback.print_exc()

## 2 & 3. Detect Scenes & Extract Per Scene

We use PySceneDetect to segment the video into semantic scenes. Then, for each scene:
- The **VLM** generates a visual caption. We enforce `presence_penalty=1.5`.
- The **ASR** extracts the transcript. We enforce `presence_penalty=2.0` (implemented as repetition penalty in standard transformers/ASR usage).

In [ ]:
def process_pipeline(video_path, output_json):
    if not os.path.exists(video_path):
        return

    corpus_data = []
    try:
        # Step 2: Detect & Split
        logger.info("=== Step 2: Splitting Video into Scenes ===")
        scenes = scene_extractor.split_video_into_scenes(video_path)
        logger.info(f"Successfully split video into {len(scenes)} scenes.")

        # Step 3: Extract Per Scene
        logger.info("=== Step 3: Extracting Visuals & Audio per Scene ===")
        for scene in scenes:
            scene_id = scene["scene_id"]
            scene_file = scene["file_path"]
            
            logger.info(f"Processing Scene {scene_id}...")
            try:
                visual_caption = vlm_processor.extract_visual_caption(scene_file)
            except RuntimeError as e:
                logger.error(f"OOM or CUDA error during VLM execution on Scene {scene_id}: {e}")
                visual_caption = "VISUAL PARSE ERROR"
                
            try:
                audio_transcript = asr_processor.extract_transcript(scene_file)
            except Exception as e:
                logger.error(f"ASR error on Scene {scene_id}: {e}")
                audio_transcript = "AUDIO PARSE ERROR"

            # Record extraction results
            corpus_data.append({
                "scene_id": scene_id,
                "start_time": scene["start_time"],
                "end_time": scene["end_time"],
                "visual_caption": visual_caption,
                "audio_transcript": audio_transcript
            })

        # Step 4: Merge to Corpus
        logger.info("=== Step 4: Merging to Corpus ===")
        save_corpus(corpus_data, output_json)
        
        # Clean up temp frames
        cleanup_temp_dir("./vea_temp_scenes")
        return corpus_data

    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        traceback.print_exc()

# Execute the pipeline
corpus_result = process_pipeline(VIDEO_PATH, OUTPUT_JSON)

## 4. Visualize Extracted Corpus

Let's view the generated corpus dataset to ensure our visual captions and textual transcripts successfully aligned per scene.

In [ ]:
if corpus_result:
    display(JSON(corpus_result))